<table>
  <tr>
    <td><div align="left"><font size="30">Dynamics of a 2-link manipulator</font></div></td>
    <td><img src="https://github.com/petercorke/robotics-toolbox-python/raw/main/docs/notebooks/figs/RobToolBox_RoundLogoB.png?raw=1" width="300"></td>
  </tr>
</table>

<p></p>

(c) Peter Corke 2020

The gravity and velocity torques of the two-link planar arm in Spong, Hutchinson and Vidyasagar, computed by the Toolbox from a DH model and from an ETS model, and compared with the analytic Euler-Lagrange equations.

> **Modernised 2026-10-19** to address API drift: this notebook was written for an earlier release of the Toolbox, and has been updated so that it runs with the current one.

![Spong book](figs/spong-2link.png)

Reference:

Robot Modeling and Control, Spong, Vidyasagar, Hutchinson, Wiley 2006

We will choose the parameters to be

In [ ]:
# RTB_BOOTSTRAP_CELL: installs the Toolbox (Colab, JupyterLite or local), generated by sync_bootstrap.py from _rtb_nb_bootstrap.py, do not edit

import subprocess
import sys

async def ensure_installed(extras: tuple[str, ...] | list[str]=()) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        await install('spatialmath-python', deps=False)
        await install('ipympl')
        if 'ik' in extras:
            await install('sympy')
        await install('roboticstoolbox-python')
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
            if 'ik' in extras:
                try:
                    import sympy
                except ImportError:
                    print("This notebook needs SymPy: pip install 'roboticstoolbox-python[ik]'")
        else:
            print('Installing roboticstoolbox-python...')
            requirement = 'roboticstoolbox-python'
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement], check=True)
            where, colab = ('on Colab', True)
    import roboticstoolbox
    version = getattr(roboticstoolbox, '__version__', 'unknown')
    print(f'Running {where} using RTB v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
from roboticstoolbox import DHRobot, RevoluteDH, Robot, Link, ET
from math import pi, sin, cos
import numpy as np

np.set_printoptions(linewidth=100, suppress=True)


# link 1
m1 = 1
l1 = 1
lc1 = 0.5

# link 2
m2 = 1
l2 = 1
lc2 = 0.5

# joint configurations
q1 = [0, 0]  # o-o- arm horizontal
q2 = [0, pi / 2]  # o-o| upper arm horizontal, lower arm vertical
q3 = [pi / 2, 0]  # o|o| arm vertical
q4 = [pi / 2, -pi / 2]  # o|o- upper arm upward, lower arm horizontal
qq = [q1, q2, q3, q4]
z = [0, 0]

# global parameters
g = 9.81

## Denavit-Hartenberg notation

In [ ]:
L1 = RevoluteDH(a=l1, m=m1, r=[-lc1, 0, 0])
L2 = RevoluteDH(a=l2, m=m2, r=[-lc2, 0, 0])
robot = DHRobot([L1, L2], gravity=[0, -g, 0])

The DH notation only allows rotation about the z-axis, which is out of the page in the Figure above.  The `gravity` argument is the acceleration due to gravity, a vector in the world frame.  Gravity acts downward, in the $-y$ direction of the figure, so we set it to `[0, -g, 0]` (the default is `[0, 0, -9.81]`).  The gravity load is the torque the joints must exert to hold the arm still against gravity, so it is positive for an arm stretched out horizontally.

In [ ]:
print(robot)
print(robot.dynamics())

In [ ]:
for q in qq:
    tau = robot.gravload(q)
    print(tau)

Using equations (7.85) and (7.86) we can write an expression for gravity torque derived analytically using Euler-Lagrange (rather than Newton-Euler) approach

In [ ]:
def EL_grav(q):
    return [
        (m1 * lc1 + m2 * l1) * g * cos(q[0]) + m2 * lc2 * g * cos(q[0] + q[1]),
        m2 * lc2 * g * cos(q[0] + q[1]),
    ]

In [ ]:
for q in qq:
    tau_rne = robot.gravload(q)
    tau_el = EL_grav(q)
    print(tau_rne, "vs", tau_el)

In fact we could use symbolic values of the constants above, repeat the above process, and generate the symbolic equations for gravity load.

Now we can explore the velocity coupling terms.  To isolate them we will set gravity and joint acceleration to zero

In [ ]:
robot.gravity = [0, 0, 0]
robot.rne(q2, [1, 1], z)

Using equations on page 261 and (7.87) we can write the velocity dependent terms as

In [ ]:
def EL_velocity(q, qd):
    h = -m2 * l1 * lc2 * sin(q[1])
    c121 = h
    c211 = c121
    c221 = h
    c112 = -h
    c122 = 0
    c222 = 0
    return [
        c121 * qd[0] * qd[1] + c211 * qd[1] * qd[0] + c221 * qd[1] ** 2,
        c112 * qd[0] ** 2,
    ]

In [ ]:
qd = [1, 1]
for q in qq:
    tau_rne = robot.rne(q, qd, z)
    tau_el = EL_velocity(q, qd)
    print(tau_rne, "vs", tau_el)

In [ ]:
from spatialmath.base import symbol

# link 1
m1 = symbol("m_1")
l1 = symbol("l_1")
lc1 = symbol("l_c_1")

# link 2
m2 = symbol("m_2")
l2 = symbol("l_2")
lc2 = symbol("l_c_2")

g = symbol("g")

L1 = RevoluteDH(a=l1, m=m1, r=[-lc1, 0, 0])
L2 = RevoluteDH(a=l2, m=m2, r=[-lc2, 0, 0])
robot = DHRobot([L1, L2], gravity=[0, -g, 0], symbolic=True)


q = symbol("q_:2")
qd = symbol("qd_:2")
robot.rne_python(q, qd, [0, 0])

## ETS notation

We will redefine the parameters of the model to numerical values

In [ ]:
# link 1
m1 = 1
l1 = 1
lc1 = 0.5

# link 2
m2 = 1
l2 = 1
lc2 = 0.5

# joint configurations
q1 = [0, 0]  # o-o- arm horizontal
q2 = [0, pi / 2]  # o-o| upper arm horizontal, lower arm vertical
q3 = [pi / 2, 0]  # o|o| arm vertical
q4 = [pi / 2, -pi / 2]  # o|o- upper arm upward, lower arm horizontal
qq = [q1, q2, q3, q4]
z = [0, 0]

# global parameters
g = 9.81

In [ ]:
L1 = Link(ets=ET.Rz(), m=1, r=[0.5, 0, 0], name="L1")
L2 = Link(ets=ET.tx(1) * ET.Rz(), m=1, r=[0.5, 0, 0], parent=L1, name="L2")
robot = Robot([L1, L2], gravity=[0, -g, 0])

In [ ]:
for q in qq:
    tau_rne = robot.gravload(q)
    tau_el = EL_grav(q)
    print(tau_rne, "vs", tau_el)